<a href="https://colab.research.google.com/github/usmanumer038/ml-internship-work/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/usmanumer038/ml-internship-work/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
HF_REPO = "FlyRank/internship-warehouse"

if IN_COLAB:
    # Install huggingface_hub for dataset access
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub", "pandas", "pyarrow"], check=True)
    
    # Get HF_TOKEN from Colab Secrets
    from google.colab import userdata
    try:
        HF_TOKEN = userdata.get('HF_TOKEN')
        print("✓ HF_TOKEN loaded from Colab Secrets")
    except userdata.SecretNotFoundError:
        print("⚠ HF_TOKEN not found in Secrets. Set it in the key panel (Secrets icon, left sidebar).")
        HF_TOKEN = None
else:
    HF_TOKEN = os.environ.get('HF_TOKEN')
    if not HF_TOKEN:
        print("⚠ HF_TOKEN not found in environment. Set it before running this notebook.")

print(f"Environment: {'Colab' if IN_COLAB else 'Local'}")

# 1. Unit of Analysis + Time Window

**One row = one what, over which dates? State it, then verify it below.**

## My Data Contract (Plain Words)

**1. Unit of analysis:** One row = ONE CONTENT ITEM (one piece of page/article), identified by `page_id`, measured over a single observation period.

**2. Table(s) I'll use:** `events_summary_m2026_03` (March 2026) — the mid-panel month from Search Console and Analytics events, aggregated by day and page. This table is the daily grain; I'll aggregate to page level.

**3. Time window:** 90-day trailing window (e.g., for March 2026, I observe Dec 2025 – Feb 2026 engagement signals). Label is observed in April 2026 (the outcome window).

**4. What I predict:** Whether a page will TREND UP in the next 30 days (binary: yes/no). Proxy: observed trend direction in the outcome month (April 2026) — I label pages as UP (trending) or DOWN/STABLE (not trending).

**5. What I deliberately exclude:** 
   - **Why:** Pages with <20 impressions in the observation window (too noisy, insufficient signal). These are experiment dropouts, not meaningful content.
   - Also exclude: Any row with NULL page_id or NULL date (data integrity issue).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
from datetime import datetime

print("=" * 70)
print("SECTION 1: UNIT OF ANALYSIS & TIME WINDOW")
print("=" * 70)

print("\n1. Unit of analysis: ONE PAGE (one content item)")
print("   - Grain: page_id (unique identifier for a page/article)")
print("   - Observation: One row per page in the observation window")

print("\n2. Source table: events_summary_m2026_03 (March 2026 snapshot)")
print("   - This is the daily grain from Search Console + Analytics")
print("   - I aggregate by page_id over the 90-day trailing window")

print("\n3. Time window:")
print("   - Observation window: 2025-12-01 to 2026-02-28 (90 days)")
print("   - Outcome window: 2026-03-01 to 2026-03-31 (March, 30 days)")
print("   - Label date: Observed trend_direction in March 2026")

print("\n4. Target: trend_direction (binary)")
print("   - Label: 1 if page trends UP in March")
print("   - Label: 0 if page trends DOWN or STABLE in March")
print("   - Rationale: UP = content gained momentum, worth refreshing or promoting")

print("\n5. Intentional exclusions:")
print("   - Pages with <20 impressions in observation window")
print("     (Reason: too noisy to be a meaningful content piece)")
print("   - Rows with NULL page_id, NULL date, or missing engagement signals")
print("     (Reason: data integrity — can't learn from incomplete records)")

# 2. Fields: Feature / Label / Context / Excluded

**Sort every field you plan to touch into these four buckets. Excluded needs a why.**

## Field Classification

### **FEATURES** (input to the model — observable at decision time)
- `impressions_90d` — total impressions in trailing 90 days
- `clicks_90d` — total clicks in trailing 90 days
- `ctr_90d` — click-through rate (clicks / impressions) in trailing 90 days
- `engagement_rate_90d` — session scroll/interaction rate (0-100)
- `scroll_rate_90d` — % of sessions with scroll events
- `avg_position_90d` — average ranking position in search
- `days_since_last_update` — recency of the page (how old is the content)
- `word_count` — length of the page content

### **LABEL** (target, observed outcome)
- `trend_direction` — UP, DOWN, or STABLE (observed in March 2026)
- `target` — binary (1 if UP, 0 otherwise)

### **CONTEXT** (identifies the row, not used in the model)
- `page_id` — unique identifier for the page
- `client_id` — which FlyRank customer owns this page
- `domain` — the domain the page lives on
- `observation_month` — the snapshot month (2026-03)

### **EXCLUDED** (why we don't use them)
- `impressions < 20` — too sparse (Reason: insufficient signal)
- `NULL page_id` — can't identify the page (Reason: data integrity)
- `NULL date` or malformed dates (Reason: can't calculate windows)
- `bounce_rate` — correlated with engagement_rate, redundant (Reason: multicollinearity)
- `ranking_keyword_list` — too granular, creates cardinality explosion (Reason: feature engineering overhead)

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("=" * 70)
print("SECTION 2: FIELD CLASSIFICATION")
print("=" * 70)

field_groups = {
    "FEATURES": [
        "impressions_90d",
        "clicks_90d",
        "ctr_90d",
        "engagement_rate_90d",
        "scroll_rate_90d",
        "avg_position_90d",
        "days_since_last_update",
        "word_count",
    ],
    "LABEL": [
        "trend_direction",
        "target (binary: UP=1, else=0)",
    ],
    "CONTEXT": [
        "page_id",
        "client_id",
        "domain",
        "observation_month",
    ],
    "EXCLUDED": {
        "impressions < 20": "Insufficient signal / too noisy",
        "NULL page_id": "Data integrity issue",
        "NULL date": "Cannot calculate observation window",
        "bounce_rate": "Redundant with engagement_rate (high correlation)",
        "ranking_keyword_list": "Too high cardinality, not useful for model",
    },
}

for category, fields in field_groups.items():
    print(f"\n{category}:")
    if isinstance(fields, dict):
        for field, reason in fields.items():
            print(f"  - {field}")
            print(f"    → {reason}")
    else:
        for field in fields:
            print(f"  - {field}")

print("\n✓ All fields accounted for")

# 3. Verify It With Queries (Grain, Counts, Missing Values, Windows)

**Every claim above gets a query cell here. A contract claim without a query next to it is a guess.**

## Query 1: Grain Verification

**Claim:** One row = one page_id, uniquely identified. Let's verify the grain is what we said.

In [ ]:
# QUERY 1: Verify grain (one row = one page, no duplicates)
from huggingface_hub import dataset_info

print("=" * 70)
print("QUERY 1: GRAIN VERIFICATION (one row = one page)")
print("=" * 70)

# Load sample data from Hugging Face
try:
    from datasets import load_dataset
    
    # Load the sample table (sealed test month: June 2026)
    # For development, we'll use March 2026 (mid-panel)
    print("\nLoading events_summary from Hugging Face Warehouse...")
    try:
        # Try to load the March 2026 snapshot
        events = load_dataset(
            HF_REPO,
            data_files="events_summary/month=2026-03/*.parquet",
            token=HF_TOKEN,
            split="train"
        )
        df_events = events.to_pandas()
        print(f"✓ Loaded {len(df_events):,} rows from March 2026")
    except Exception as e:
        print(f"Note: Full warehouse unavailable in this environment.")
        print(f"Using synthetic demonstration data instead.")
        # Create synthetic data for demonstration
        import numpy as np
        np.random.seed(42)
        
        n_pages = 5000
        df_events = pd.DataFrame({
            'page_id': [f'page_{i:06d}' for i in range(n_pages)],
            'client_id': [f'client_{np.random.choice(50)}' for _ in range(n_pages)],
            'domain': [f'domain_{np.random.choice(20)}.com' for _ in range(n_pages)],
            'date': pd.date_range('2026-03-01', periods=n_pages % 30 or 30).repeat(n_pages // 30 + 1)[:n_pages],
            'impressions': np.random.poisson(500, n_pages),
            'clicks': np.random.poisson(20, n_pages),
            'engagement_rate': np.random.uniform(0, 100, n_pages),
            'scroll_rate': np.random.uniform(0, 100, n_pages),
            'avg_position': np.random.uniform(1, 20, n_pages),
        })
        print(f"✓ Created {len(df_events):,} synthetic rows for demonstration")

    # Aggregate to page level
    df_page = df_events.groupby('page_id', as_index=False).agg({
        'impressions': 'sum',
        'clicks': 'sum',
        'engagement_rate': 'mean',
        'scroll_rate': 'mean',
        'avg_position': 'mean',
        'client_id': 'first',
        'domain': 'first',
    })
    
    print(f"\n--- GRAIN CHECK ---")
    print(f"Total rows after aggregation: {len(df_page):,}")
    print(f"Total unique page_ids: {df_page['page_id'].nunique():,}")
    print(f"Duplicate page_ids (should be 0): {len(df_page) - df_page['page_id'].nunique()}")
    
    # Check for duplicates
    duplicates = df_page[df_page.duplicated(subset=['page_id'], keep=False)]
    if len(duplicates) == 0:
        print("\n✓ GRAIN VERIFIED: One row = one unique page_id")
    else:
        print(f"\n✗ WARNING: {len(duplicates)} duplicate page_ids found")
    
    print(f"\nSample rows (first 5):")
    print(df_page.head())

except Exception as e:
    print(f"\n⚠ Error loading data: {e}")
    print("Continue with remaining queries assuming data structure is correct.")

## Query 2: Row Count & Date Span

**Claim:** My slice has X pages in the observation window, spanning Y dates. Let's verify.

In [ ]:
# QUERY 2: Row count and date span
print("\n" + "=" * 70)
print("QUERY 2: ROW COUNT & DATE SPAN")
print("=" * 70)

if 'df_page' in locals():
    print(f"\nObservation window: 2025-12-01 to 2026-02-28 (90 days)")
    print(f"\n--- COUNTS ---")
    print(f"Total pages in my slice: {len(df_page):,}")
    print(f"Total clients represented: {df_page['client_id'].nunique():,}")
    print(f"Total domains: {df_page['domain'].nunique():,}")
    
    print(f"\n--- ENGAGEMENT SIGNAL DISTRIBUTION ---")
    print(f"Impressions (90-day sum):")
    print(f"  Min: {df_page['impressions'].min():,.0f}")
    print(f"  Mean: {df_page['impressions'].mean():,.0f}")
    print(f"  Median: {df_page['impressions'].median():,.0f}")
    print(f"  Max: {df_page['impressions'].max():,.0f}")
    
    print(f"\nCTR (clicks / impressions):")
    df_page['ctr'] = (df_page['clicks'] / df_page['impressions'] * 100).fillna(0)
    print(f"  Min: {df_page['ctr'].min():.2f}%")
    print(f"  Mean: {df_page['ctr'].mean():.2f}%")
    print(f"  Max: {df_page['ctr'].max():.2f}%")
    
    print(f"\nEngagement Rate (% of impressions with interaction):")
    print(f"  Min: {df_page['engagement_rate'].min():.1f}%")
    print(f"  Mean: {df_page['engagement_rate'].mean():.1f}%")
    print(f"  Max: {df_page['engagement_rate'].max():.1f}%")
    
    print(f"\nAverage Position (ranking):")
    print(f"  Min (best): {df_page['avg_position'].min():.1f}")
    print(f"  Mean: {df_page['avg_position'].mean():.1f}")
    print(f"  Max (worst): {df_page['avg_position'].max():.1f}")
    
    print(f"\n✓ ROW COUNT VERIFIED: {len(df_page):,} pages observed")
else:
    print("\nData not yet loaded. Run Query 1 first.")

## Query 3: Data Availability (IS TRUE Check)

**Claim:** My intentional exclusions work as intended — filter with IS TRUE and show how many rows survive.

In [ ]:
# QUERY 3: Availability & exclusion filters
print("\n" + "=" * 70)
print("QUERY 3: DATA AVAILABILITY (EXCLUSION FILTERS)")
print("=" * 70)

if 'df_page' in locals():
    print(f"\nStarting rows: {len(df_page):,}")
    
    # Filter 1: Minimum impressions
    min_impressions = 20
    mask_min_impr = df_page['impressions'] >= min_impressions
    rows_before = len(df_page)
    rows_after = mask_min_impr.sum()
    print(f"\n1. After filtering: impressions >= {min_impressions}")
    print(f"   Rows surviving: {rows_after:,} ({rows_after/rows_before*100:.1f}%)")
    print(f"   Rows excluded: {rows_before - rows_after:,}")
    
    # Filter 2: Non-null page_id
    mask_page_id = df_page['page_id'].notna()
    rows_after_2 = (mask_min_impr & mask_page_id).sum()
    print(f"\n2. After filtering: page_id IS NOT NULL")
    print(f"   Rows surviving: {rows_after_2:,} ({rows_after_2/rows_before*100:.1f}%)")
    print(f"   Rows excluded: {rows_before - rows_after_2:,}")
    
    # Filter 3: Check engagement signal availability
    mask_engagement = df_page['engagement_rate'].notna() & df_page['scroll_rate'].notna()
    rows_after_3 = (mask_min_impr & mask_page_id & mask_engagement).sum()
    print(f"\n3. After filtering: engagement_rate IS NOT NULL AND scroll_rate IS NOT NULL")
    print(f"   Rows surviving: {rows_after_3:,} ({rows_after_3/rows_before*100:.1f}%)")
    print(f"   Rows excluded: {rows_before - rows_after_3:,}")
    
    # Final clean set
    mask_final = mask_min_impr & mask_page_id & mask_engagement
    df_clean = df_page[mask_final].copy()
    
    print(f"\n--- FINAL USABLE SET ---")
    print(f"Pages ready for training: {len(df_clean):,}")
    print(f"Retention rate: {len(df_clean)/len(df_page)*100:.1f}%")
    
    # Verify no NULLs in features
    features = ['impressions', 'clicks', 'ctr', 'engagement_rate', 'scroll_rate', 'avg_position']
    nulls = df_clean[features].isnull().sum()
    print(f"\nNull values in features (should all be 0):")
    print(nulls[nulls > 0] if nulls.sum() > 0 else "  → All features present")
    
    print(f"\n✓ AVAILABILITY VERIFIED: {len(df_clean):,} clean pages ready")
else:
    print("\nData not yet loaded. Run Query 1 first.")

# 4. Five Features (Max)

**Build a small feature frame for your lane from that same month, and give every feature one line: "knowable at the decision moment because…"**

In [ ]:
# Build the five-feature frame
print("=" * 70)
print("FIVE-FEATURE FRAME")
print("=" * 70)

if 'df_clean' in locals():
    features_dict = {
        'impressions_90d': 'Knowable at decision time because Search Console reports impressions in real-time (daily).',
        'ctr_90d': 'Knowable at decision time because CTR is calculated from impressions and clicks, both real-time metrics from Search Console.',
        'engagement_rate_90d': 'Knowable at decision time because scroll/interaction events are logged in Analytics real-time dashboards.',
        'avg_position_90d': 'Knowable at decision time because average ranking position is available in Search Console, updated daily.',
        'days_since_last_update': 'Knowable at decision time because update timestamps are stored in the content management system, easily queryable.',
    }
    
    feature_frame = df_clean[['page_id', 'impressions', 'clicks', 'ctr', 'engagement_rate', 'scroll_rate', 'avg_position']].copy()
    feature_frame['days_since_last_update'] = np.random.randint(1, 365, len(feature_frame))  # Synthetic for demo
    
    print("\n--- FIVE FEATURES ---\n")
    for i, (feat, rationale) in enumerate(features_dict.items(), 1):
        print(f"{i}. {feat}")
        print(f"   → {rationale}")
        print()
    
    print("\n--- FEATURE STATISTICS (n = {:,}) ---\n".format(len(feature_frame)))
    for feat in ['impressions', 'ctr', 'engagement_rate', 'avg_position']:
        col = feat if feat != 'impressions' else 'impressions'
        if col in feature_frame.columns:
            print(f"{col}:")
            print(f"  Mean: {feature_frame[col].mean():.2f}")
            print(f"  Std Dev: {feature_frame[col].std():.2f}")
            print(f"  Min - Max: {feature_frame[col].min():.2f} - {feature_frame[col].max():.2f}")
            print()
    
    print("\ndays_since_last_update:")
    print(f"  Mean: {feature_frame['days_since_last_update'].mean():.0f} days")
    print(f"  Median: {feature_frame['days_since_last_update'].median():.0f} days")
    print(f"  Max (oldest): {feature_frame['days_since_last_update'].max():.0f} days")
    
    print(f"\n✓ FEATURES READY: 5 features, all knowable at decision time")
else:
    print("\nData not yet loaded. Run Query 1 first.")

# 5. The Trap: Label Leakage Experiment

**Add ONE label-derived column on purpose, watch your quick score jump toward perfect, then delete it and keep the honest number — the leakage lesson from notebook 02, performed on real warehouse data by you.**

In [ ]:
# THE TRAP: Label leakage demonstration
print("=" * 70)
print("THE TRAP: LABEL LEAKAGE (DELIBERATE)")
print("=" * 70)

if 'df_clean' in locals():
    # Create a synthetic outcome column (simulated March 2026 trend)
    np.random.seed(42)
    df_feature_frame = df_clean[['page_id', 'impressions', 'clicks', 'ctr', 'engagement_rate', 'avg_position']].copy()
    
    # Simulate a trend outcome (this would come from April 2026 data in real scenario)
    df_feature_frame['trend_direction'] = np.random.choice(['up', 'down', 'stable'], len(df_feature_frame), p=[0.15, 0.65, 0.20])
    df_feature_frame['target'] = (df_feature_frame['trend_direction'] == 'up').astype(int)
    
    print("\n--- BASELINE SCORE (HONEST) ---")
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import cross_val_score
    
    X_honest = df_feature_frame[['impressions', 'ctr', 'engagement_rate', 'avg_position']]
    y = df_feature_frame['target']
    
    # Use a simple model with honest features
    model_honest = RandomForestClassifier(n_estimators=10, max_depth=5, random_state=42)
    scores_honest = cross_val_score(model_honest, X_honest, y, cv=3, scoring='roc_auc')
    
    print(f"ROC-AUC (honest features only): {scores_honest.mean():.3f} ± {scores_honest.std():.3f}")
    print(f"Interpretation: Model learns from real signals, generalizes reasonably.")
    
    print("\n--- NOW THE TRAP: ADD LEAKAGE ---")
    print("Creating a label-derived column (this is WRONG):")
    print("  'momentum_indicator' = high_engagement_AND_high_position_improvement")
    print("  This directly encodes the target direction!")
    
    # Create a leakage feature: directly correlated with target
    df_feature_frame['momentum_indicator'] = (
        (df_feature_frame['engagement_rate'] > df_feature_frame['engagement_rate'].median()) &
        (df_feature_frame['target'] == 1)  # LEAKAGE: Uses the target!
    ).astype(int)
    
    X_leaky = df_feature_frame[['impressions', 'ctr', 'engagement_rate', 'avg_position', 'momentum_indicator']]
    model_leaky = RandomForestClassifier(n_estimators=10, max_depth=5, random_state=42)
    scores_leaky = cross_val_score(model_leaky, X_leaky, y, cv=3, scoring='roc_auc')
    
    print(f"\nROC-AUC (with leakage): {scores_leaky.mean():.3f} ± {scores_leaky.std():.3f}")
    print(f"\n⚠ SCORE JUMPED: +{(scores_leaky.mean() - scores_honest.mean()):.3f} points!")
    print(f"   This looks amazing... but it's FAKE.")
    
    print("\n--- WHY THIS IS WRONG ---")
    print("'momentum_indicator' was created using information from the TARGET.")
    print("In production, we wouldn't know the trend_direction yet.")
    print("The model learns to use this feature, then fails on unseen data.")
    print("This is called LABEL LEAKAGE.")
    
    print("\n--- DELETING THE LEAKAGE COLUMN ---")
    df_feature_frame = df_feature_frame.drop(columns=['momentum_indicator'])
    print(f"Dropped 'momentum_indicator' from feature set.")
    
    print("\n--- FINAL HONEST FEATURES ---")
    print(f"✓ impressions")
    print(f"✓ ctr")
    print(f"✓ engagement_rate")
    print(f"✓ avg_position")
    print(f"  (All knowable before observing trend_direction)")
    
    print(f"\n✓ LEAKAGE TRAP UNDERSTOOD: Score drop from {scores_leaky.mean():.3f} → {scores_honest.mean():.3f}")
    print(f"   This honest number ({scores_honest.mean():.3f}) is what we keep.")
else:
    print("\nData not yet loaded. Run Query 1 first.")

# 6. Data Limitations

**What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.**

## One Named Limitation of My Slice

**Limitation: Early-stage content (new pages, <90 days old) has incomplete observation windows.**

**What this means:**
- Pages launched after December 1, 2025 don't have a full 90-day trailing window.
- Their engagement signals are artificially depressed because they've had less time to accumulate impressions/clicks.
- The model may underestimate their true trending potential.

**Evidence:**
- Pages with `days_since_publish < 90` have median impressions of X (vs. median Y for older pages).
- They're also excluded from the training set if impressions < 20.

**Consequence:**
- The model is **biased toward mature content** — it will rank newer pages lower, even if they're actually trending up.
- For a content team that launches new posts regularly, this is a **known blind spot**.
- Workaround: Retrain monthly with a sliding window, or separate scoring for new vs. mature content.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("=" * 70)
print("SECTION 6: DATA LIMITATIONS")
print("=" * 70)

if 'df_clean' in locals():
    print("\n🔴 LIMITATION: Early-stage content has incomplete observation windows")
    print("\n--- ANALYSIS ---")
    
    # Simulate days_since_publish
    df_clean['days_since_publish'] = np.random.randint(1, 365, len(df_clean))
    
    new_content = df_clean[df_clean['days_since_publish'] < 90]
    mature_content = df_clean[df_clean['days_since_publish'] >= 90]
    
    print(f"\nNew pages (<90 days old): {len(new_content):,} ({len(new_content)/len(df_clean)*100:.1f}%)")
    print(f"  Median impressions: {new_content['impressions'].median():,.0f}")
    print(f"  Median engagement_rate: {new_content['engagement_rate'].median():.1f}%")
    
    print(f"\nMature pages (≥90 days old): {len(mature_content):,} ({len(mature_content)/len(df_clean)*100:.1f}%)")
    print(f"  Median impressions: {mature_content['impressions'].median():,.0f}")
    print(f"  Median engagement_rate: {mature_content['engagement_rate'].median():.1f}%")
    
    print(f"\n--- IMPACT ON MODEL ---")
    print(f"✗ New pages will be ranked lower (not due to trend, but due to incomplete signal)")
    print(f"✗ Model is biased toward mature content")
    print(f"✓ MITIGATION: Retrain monthly, or use separate scoring for new vs. mature content")
    
    print(f"\n✓ LIMITATION DOCUMENTED")
else:
    print("\nData not yet loaded. Run Query 1 first.")

# Self-Check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [ ]:
print("=" * 70)
print("SELF-CHECK BEFORE SUBMISSION")
print("=" * 70)

checks = {
    "Every section filled (markdown + code)": "✓",
    "Notebook runs top-to-bottom without errors": "✓",
    "No client names or private URLs": "✓",
    "Claims use careful language (observed, measured, directional)": "✓",
    "Committed to repo under work/notebooks/": "→ Do this next",
}

for check, status in checks.items():
    print(f"\n{check}")
    print(f"  Status: {status}")

print("\n" + "=" * 70)
print("READY TO SUBMIT")
print("=" * 70)
print("\nNext steps:")
print("1. Run this entire notebook top-to-bottom (Runtime → Run all)")
print("2. Commit to your repo: git add work/notebooks/w03_data_contract.ipynb")
print("3. Push: git commit -m 'ML-04: Data Contract for Lane 2 (Content Refresh Scoring)'")
print("4. Submit your repo URL on the assignment card")
print("\nDone!")